# D2 · M2.1 — LangChain 1.0 Core

**Worked side by side with the Concept HTML page, not read start to finish.** Read a short
concept section on the page, run the matching task below, then go back to the page for the next
concept — four round trips, not one long lab.

**THE SITUATION:** M1.2's theory covered turning a messy customer message into structured data
(name / account type / intent / urgency / summary) using tool calling. The lab that would have
built it never happened (VM issue on Day 1). This notebook shows it built for real — same job,
in LangChain, the framework the rest of this programme runs on.

**This notebook is fully working code — nothing to write.** Run each cell in order, read what
it prints, and follow the concept page's lab-marker cues to know when to come back here.

Concept page: `Day2/concept/D2_M2.1_LangChain_Core_Concept.html`


## Setup

Loads your API key and the schema shared with M1.2. Run this once before anything else.


In [1]:
import json
import os
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

# Loads OPENAI_API_KEY from the .env file sitting next to this notebook.
# Find the .env file wherever it lives: this folder, any folder above it, or the
# Day1/labs/starter/ folder SETUP.md originally told you to create it in. One key
# file for the whole repo -- you never need to copy it between Day folders.
def find_env():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / ".env").is_file():
            return candidate / ".env"
        legacy = candidate / "Day1" / "labs" / "starter" / ".env"
        if legacy.is_file():
            return legacy
    return None


_env = find_env()
if _env:
    load_dotenv(_env)
    print(f"Loaded API key from: {_env}")
else:
    print("WARNING: no .env file found -- see SETUP.md Step 2.")

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set. Set it before continuing — never hard-code a key in this notebook."
)

# Same 10 messages M1.2 would have used — reused on purpose from Day 1's data folder, so
# today's result is directly comparable to what M1.2's theory described.
DATA_PATH = Path.cwd().resolve().parents[2] / "Day1" / "data" / "D1_M1.2_sample_customer_messages.json"


class CustomerRequest(BaseModel):
    """Structured record extracted from a free-text Meridian Retail Bank customer message."""

    name: str = Field(
        ..., min_length=1,
        description="Customer's name if stated or clearly implied; 'Unknown' if not given.",
    )
    account_type: Literal[
        "debit_card", "credit_card", "savings_account",
        "personal_loan", "mortgage", "unknown",
    ] = Field(..., description="The Meridian Retail Bank account/product type the message is about.")
    intent: Literal[
        "card_issue", "account_access", "loan_inquiry",
        "complaint", "general_question", "fraud_report",
    ] = Field(..., description="The primary reason the customer is reaching out.")
    urgency: Literal["low", "medium", "high"] = Field(
        ..., description="How time-sensitive this request is, based on the message's own wording."
    )
    customer_summary: str = Field(
        ..., min_length=10, description="One-sentence, human-readable summary of what the customer needs."
    )

print("Setup complete — CustomerRequest schema and sample-data path ready.")


Loaded API key from: C:\Users\Administrator\Training\EY_GenAI_3D\Day1\labs\starter\.env
Setup complete — CustomerRequest schema and sample-data path ready.


## Concept A, on the page — then come back here for Task 1

Read **Concept A — One Interface, Any Provider** on the concept page, try the provider toggle,
then run the cell below.


In [2]:
# TASK 1 — init_chat_model: LangChain's provider-agnostic model interface. Same call shape
# works for OpenAI, Anthropic, or any other supported provider — only the model name/provider
# string changes, never your calling code. Nothing to edit here — just run it and read the output.
model = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)

print("=" * 72)
print("TASK 1 — init_chat_model: one interface, any provider")
print("=" * 72)
response = model.invoke("Say hello to a Meridian Retail Bank customer in one short sentence.")
print(f"\nModel replied: {response.content}")
print(
    "\nIf this were Anthropic instead of OpenAI, only the model string and "
    "model_provider argument above would change — every line below this still works."
)


TASK 1 — init_chat_model: one interface, any provider

Model replied: Hello, valued Meridian Retail Bank customer!

If this were Anthropic instead of OpenAI, only the model string and model_provider argument above would change — every line below this still works.


**Notice:** the reply came from a real call, but nothing above named an OpenAI-specific
function — that's the whole point of Task 1. Now go back to the concept page for **Concept B**.


## Concept B, on the page — then come back here for Tasks 2, 3 and 4

Read **Concept B — Composing The Chain**, click through the four stages, then run the cell
below — it builds the exact same chain you just clicked through, for real, and tests it on one
message.


In [3]:
# TASK 2 — The prompt template. Read this — nothing to edit. Same extraction job M1.2's
# theory described, written as a reusable, parameterised template instead of an inline string.
EXTRACTION_SYSTEM_PROMPT = (
    "You are a request-triage assistant for Meridian Retail Bank. Read the customer's "
    "message and extract: their name (or 'Unknown' if not stated), the account/product "
    "type, their intent, how urgent it is, and a one-sentence summary."
)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", EXTRACTION_SYSTEM_PROMPT),
    ("user", "{message}"),
])

# TASK 3 — Compose the extraction chain: pipe the prompt into a structured-output-bound model.
# This one line replaces the manual tool-call JSON parsing M1.2's theory described —
# LangChain's with_structured_output does that parsing and validation for you.
extraction_chain = extraction_prompt | model.with_structured_output(CustomerRequest)

# TASK 4 — Add retry. M1.2's theory described a manual while-loop retrying on a bad response —
# LangChain does the same job in one line.
extraction_chain = extraction_chain.with_retry(stop_after_attempt=3)


def extract_customer_request(message: str) -> CustomerRequest:
    """Runs the composed chain end to end: prompt -> model -> validated CustomerRequest."""
    result = extraction_chain.invoke({"message": message})
    if not isinstance(result, CustomerRequest):
        raise TypeError(
            "extraction_chain didn't return a CustomerRequest — check the chain composition above."
        )
    return result


print("=" * 72)
print("TASKS 2-4 — the chain, composed and retry-wrapped — one test message")
print("=" * 72)
test_result = extract_customer_request(
    "Hi, this is Priya. My mortgage application seems stuck and I need an update urgently."
)
print(f"\nOne test message -> {test_result.model_dump_json(indent=2)}")


TASKS 2-4 — the chain, composed and retry-wrapped — one test message

One test message -> {
  "name": "Priya",
  "account_type": "mortgage",
  "intent": "loan_inquiry",
  "urgency": "high",
  "customer_summary": "Priya is seeking an urgent update on her stuck mortgage application."
}


**Notice:** the chain above turned one free-text sentence into a validated, typed record
in three lines of setup. Now go back to the concept page for **Concept C**.


## Concept C, on the page — then come back here for Task 5

Read **Concept C — 20 Lines Become 2**, toggle between the manual and chain versions, then run
the cell below — the same chain against all 10 real sample messages. **This is the module's
graded deliverable.**


In [4]:
def load_sample_messages() -> list:
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


# TASK 5 — Run the chain against the same 10 messages M1.2 would have used. This IS the
# module's graded deliverable: a working extraction chain that holds up across real messages,
# not just the one example above.
def run_extraction_evaluation(messages):
    print("=" * 72)
    print("TASK 5 — EXTRACTION CHAIN vs. THE 10 SAMPLE MESSAGES")
    print("=" * 72)
    report = []
    for item in messages:
        try:
            result = extract_customer_request(item["message"])
            report.append({"id": item["id"], "ok": True, "result": result.model_dump()})
            print(f"\n[{item['id']}] {item['message']}")
            print(f"  -> {result.model_dump_json()}")
        except Exception as e:
            report.append({"id": item["id"], "ok": False, "error": f"{type(e).__name__}: {e}"})
            print(f"\n[{item['id']}] {item['message']}")
            print(f"  -> FAILED: {type(e).__name__}: {e}")
    passed = sum(1 for r in report if r["ok"])
    print(f"\n{passed}/{len(report)} messages extracted successfully.")
    return report


sample_messages = load_sample_messages()
evaluation_report = run_extraction_evaluation(sample_messages)


TASK 5 — EXTRACTION CHAIN vs. THE 10 SAMPLE MESSAGES

[1] Hi, my debit card was declined twice today even though I have money in my account. This is urgent, I need to pay rent this afternoon.
  -> {"name":"Unknown","account_type":"debit_card","intent":"card_issue","urgency":"high","customer_summary":"The customer is experiencing issues with their debit card being declined and needs immediate assistance to resolve it before paying rent."}

[2] I noticed a transaction on my credit card statement for $340 that I did not make. Please look into this immediately.
  -> {"name":"Unknown","account_type":"credit_card","intent":"fraud_report","urgency":"high","customer_summary":"The customer is reporting an unauthorized transaction on their credit card and requests immediate investigation."}

[3] Can you tell me what documents I need to apply for a personal loan of around 5 lakhs?
  -> {"name":"Unknown","account_type":"personal_loan","intent":"loan_inquiry","urgency":"medium","customer_summary":"

**Notice:** the pass/fail count above — which messages (if any) failed, and why. Now go
back to the concept page for **Concept D**.


## Concept D, on the page — then come back here for Task 6

Read **Concept D — The Agent Decides, You Don't**, click both questions, then run the cell
below — a real minimal agent making the same kind of choice.


In [5]:
# TASK 6 — READ-ALONG DEMO (not graded, nothing to edit). A minimal tool-using agent with
# create_agent — the abstraction this module introduces. Full hands-on agent-building (multiple
# tools, tracing your own reasoning) is Day 3's M3.1 — today is just seeing the shape of it once.
@tool
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, e.g. '12 * 4' or '2500 / 20'."""
    return str(eval(expression, {"__builtins__": {}}, {}))


agent = create_agent(model, tools=[calculator])

print("=" * 72)
print("TASK 6 — READ-ALONG: a minimal tool-calling agent (create_agent)")
print("=" * 72)
agent_result = agent.invoke({"messages": [{"role": "user", "content": "What is 2500 divided by 20?"}]})
last_message = agent_result["messages"][-1]
print(f"\nAgent's final answer: {last_message.content}")
print(
    "\nNotice: create_agent decided on its own to call the calculator tool instead of guessing "
    "the arithmetic itself. Day 3's M3.1 is where you build this yourself, with more than one "
    "tool to choose between."
)


TASK 6 — READ-ALONG: a minimal tool-calling agent (create_agent)

Agent's final answer: 2500 divided by 20 is 125.0.

Notice: create_agent decided on its own to call the calculator tool instead of guessing the arithmetic itself. Day 3's M3.1 is where you build this yourself, with more than one tool to choose between.


## Recap, on the page — then save your results

Read the **Recap** section on the concept page, then run the final cell below — it saves
`m2_1_extraction_results.json` next to this notebook, which the autograder reads.


In [6]:
key_takeaway = """
KEY TAKEAWAY: with_structured_output replaced M1.2's manual tool-schema
dict + raw tool_calls parsing + Pydantic validation try/except with one
method call. with_retry replaced a hand-written for-loop with backoff
with one more. Two lines doing the job ~20 lines used to do.
"""
print(key_takeaway)

results = {
    "sample_evaluation": evaluation_report,
    "key_takeaway": key_takeaway,
}

with open("m2_1_extraction_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved m2_1_extraction_results.json —", len(json.dumps(results)), "bytes")



KEY TAKEAWAY: with_structured_output replaced M1.2's manual tool-schema
dict + raw tool_calls parsing + Pydantic validation try/except with one
method call. with_retry replaced a hand-written for-loop with backoff
with one more. Two lines doing the job ~20 lines used to do.

Saved m2_1_extraction_results.json — 2978 bytes
